In [0]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder \
    .appName("MySparkApp") \
    .master("local[*]") \
    .getOrCreate()

# Access SparkContext from SparkSession
sc = spark.sparkContext

In [0]:
df= spark.read.format('csv').option('inferSchema',True).option('header',True).load('/users/shubh/Data Engineering/BigMart Sales.csv')

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df_json = spark.read.format('json').option('inferedSchema', True)\
    .option('header', True)\
        .option('multiLine',False).load('/users/shubh/Data Engineering/drivers.json')

In [0]:
df.filter(col('Outlet_Location_Type').isin('Tier 1','Tier 2') & col('Outlet_Size').isNull()).limit(20).toPandas()

In [0]:
df.limit(20).toPandas()

In [0]:
df.show()

### Column Rename: withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt').limit(10).toPandas()

### withColumn

## Scenario 1

In [0]:
df_new = df.withColumn('Flag',lit('New'))

In [0]:
df_new.limit(10).toPandas()

In [0]:
df_new.withColumn('Multiple', col('Item_Weight')*col('Item_MRP')).toPandas()

### Scenario 2

In [0]:
df_new.withColumn('Item_Fat_Content', regexp_replace('Item_Fat_Content','Regular','Reg'))\
        .withColumn('Item_Fat_Content', regexp_replace('Item_Fat_Content','Low Fat','LF')).toPandas()

### Type Casting

In [0]:
df_new1= df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))

In [0]:
df_new1.printSchema()

# Sort

## Scenario 1

In [0]:
df.sort(col('Item_Weight').desc()).limit(50).toPandas()

In [0]:
df.sort(col('Item_Visibility').asc()).limit(10).toPandas()

## Scenario 2

In [0]:
df.sort(['Item_Weight','Item_MRP'], ascending= [0,1]).limit(100).toPandas()

# limit

In [0]:
df.limit(10).show()

# DROP

In [0]:
df.drop('Item_Visibility').limit(10).show()

In [0]:
df.limit(10).toPandas()

In [0]:
df.drop('Item_Visibility','Outlet_Type').limit(10).toPandas()

# Drop Duplicates

### Scenario 1 (table level)

In [0]:
df.toPandas()

In [0]:
df.dropDuplicates().toPandas()

In [0]:
df.distinct().toPandas()

### Scenario 2 (column level)

In [0]:
df.drop_duplicates(subset=['Item_Type']).toPandas()

# Union

### creating Dataframes

In [0]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("example").getOrCreate()

# Data and schema for df1
data1 = [('1', 'kad'), ('2', 'sid')]
schema1 = 'id STRING, name STRING'
df1 = spark.createDataFrame(data1, schema1)

# Data and schema for df2
data2 = [('3', 'rahul'), ('4', 'jas')]
schema2 = 'id STRING, name STRING'
df2 = spark.createDataFrame(data2, schema2)

# Show the dataframes
df1.show()
df2.show()

In [0]:
df1.union(df2).show()

In [0]:
data1 = [('kad','1'), ('sid','2')]
schema1 = 'name STRING,id STRING'
df1 = spark.createDataFrame(data1, schema1)

# Data and schema for df2
data2 = [('3', 'rahul'), ('4', 'jas')]
schema2 = 'id STRING, name STRING'
df2 = spark.createDataFrame(data2, schema2)

# Show the dataframes
df1.show()
df2.show()
df1.union(df2).show()

### unionByName

In [0]:
df1.unionByName(df2).show()

In [0]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.range(5).show()

# String Functions

In [0]:
df.toPandas()

In [0]:
df.select(initcap('Item_Type').alias('Item_Type')).toPandas()

In [0]:
df.select(upper('Item_Type').alias('Item_Type')).toPandas()

In [0]:
df.select(lower('Item_Type').alias('Item_Type')).toPandas()

# Data functions

In [0]:
df= df.withColumn('curr_date',current_date())
df.toPandas()

In [0]:
df= df.withColumn('week_after',date_add('curr_date',7))
df.toPandas()

### date substract

In [0]:
df= df.withColumn('week_before',date_sub('curr_date',7))
df.toPandas()

In [0]:
df= df.withColumn('week_before',date_add('curr_date',-7))
df.toPandas()

In [0]:
df=df.withColumn('date_diff',date_diff('week_after','week_before'))
df.toPandas()

## Date format

In [0]:
df = df.withColumn('week_before', date_format('week_before', 'dd-MM-yyyy'))
df.toPandas()

In [0]:
df.printSchema()

In [0]:
df.toPandas()


# Handling nulls
### Droping nulls

In [0]:
df.dropna('all').toPandas() # all colums null

In [0]:
df.dropna('any').toPandas()  #any column is null

In [0]:
df.dropna(subset=['Outlet_Size']).toPandas() #specific column

### Filling Nulls

In [0]:
df.fillna('Not available').toPandas()

In [0]:
df.fillna('Not Available',subset=['Outlet_Size']).show()

# Split and indexing
### Split

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type', ' ')).show()

### indexing

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type', ' ')[1]).show()

# Explode

In [0]:
df_exp =df.withColumn('Outlet_Type',split('Outlet_Type', ' '))

df_exp.toPandas()

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).toPandas()

### array contains

In [0]:
df_exp.withColumn('Outlet_Type_flag',array_contains('Outlet_Type','Type1')).toPandas()

# Group by

### Scenario 1

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).toPandas()

In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP')).toPandas()

### Scenario 2

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Item_MRP_sum')).toPandas()

### Scenario 3

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Item_MRP_sum'),avg('Item_MRP')\
                                          .alias('Item_MRP_Average')).toPandas()

### collect list

In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.show()

In [0]:
df_book.groupBy('user').agg(collect_list('book')).show()

In [0]:
df.select('Item_Type','Outlet_Size','Item_MRP').show()

# Pivot

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).show()